# 10 · FMHA — a multi-role pipeline

Every kernel so far has been a GEMM. Flash attention is a harder case: more roles, a softmax between two matmuls, and a producer-consumer structure that spans the whole pipeline.

It is also the best demonstration of why per-warp tracing matters. Averaged over the kernel, attention looks matmul-bound. It is not.

## 1. The question

The public CUTLASS `fmha.py` specialises **sixteen warps** into six roles:

| Role | Warps | Job |
|---|---|---|
| `load` | 1 | TMA loads of Q, K, V |
| `mma` | 1 | QK and PV matmuls |
| `softmax0` | 4 | Softmax, first stage |
| `softmax1` | 4 | Softmax, second stage |
| `correction` | 4 | Running-max/sum rescaling |
| `epilogue` | 1 | Write O out |

**The question:** where does the time actually go — the matmuls, or everything else?

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        610.57.04
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

This kernel was instrumented **for this notebook**, from the unmodified public `fmha.py`, by wrapping each of the six role-dispatch bodies in one range.

Role-level granularity is a deliberate choice. `fmha.py` is 4,500 lines; instrumenting every wait inside it would produce an event volume that risks overflowing the trace buffer — the failure mode described in notebook 11, where you get an empty trace and no error. Six ranges answer the question at hand.

**Start coarse. Add detail only where the first profile shows a problem.**

In [2]:
nbhelp.show_iket_calls(
    nbhelp.kernel("instrumented", "fmha_iket_roles.py"),
    context=1,
)

fmha_iket_roles.py: 12 IKET call sites
------------------------------------------------------------------------
  1057          if warp_idx == self.load_warp_id:
> 1058              cute.experimental.iket.range_push("load_block")
  1059              cute.arch.setmaxregister_decrease(self.num_regs_other)
       ...
  1224                  # End of persistent scheduler loop
> 1225              cute.experimental.iket.range_pop()  # load_block
  1226          # ///////////////////////////////////////////////////////////////////////////////
       ...
  1229          if warp_idx == self.mma_warp_id:
> 1230              cute.experimental.iket.range_push("mma_block")
  1231              cute.arch.setmaxregister_decrease(self.num_regs_other)
       ...
  1427              # End of persistent scheduler loop
> 1428              cute.experimental.iket.range_pop()  # mma_block
  1429          # ///////////////////////////////////////////////////////////////////////////////
       ...
  1432       

Confirm it changed nothing but instrumentation — the diff against the pristine upstream file should be six pushes and six pops and nothing else:

In [3]:
nbhelp.show_instrumentation_diff(
    nbhelp.kernel("upstream", "fmha.py"),
    nbhelp.kernel("instrumented", "fmha_iket_roles.py"),
    limit=50,
)

added 12 lines, removed 0
every added line is an IKET call: instrumentation only.
------------------------------------------------------------------------
--- fmha.py
+++ fmha_iket_roles.py
@@ -1055,6 +1055,7 @@
         #  LOAD
         # ///////////////////////////////////////////////////////////////////////////////
         if warp_idx == self.load_warp_id:
+            cute.experimental.iket.range_push("load_block")
             cute.arch.setmaxregister_decrease(self.num_regs_other)
             cute.arch.griddepcontrol_wait()
             while work_tile.is_valid_tile:
@@ -1221,10 +1222,12 @@
                 tile_sched.advance_to_next_work()
                 work_tile = tile_sched.get_current_work()
                 # End of persistent scheduler loop
+            cute.experimental.iket.range_pop()  # load_block
         # ///////////////////////////////////////////////////////////////////////////////
         #  MMA
         # /////////////////////////////////////////////////////

## 3. Run it

`fmha.py` imports shared helpers, so the vendored `upstream/` directory has to be on `PYTHONPATH`.

In [4]:
import os
os.environ["PYTHONPATH"] = nbhelp.kernel("upstream") + os.pathsep + os.environ.get("PYTHONPATH", "")

cmd = (
    "run-iket --output-dir out/nb10_fmha --clobber profile --postprocess all -- "
    f"python {nbhelp.kernel('instrumented', 'fmha_iket_roles.py')}"
)
check_env.run_or_show(cmd, env, est="about 2 minutes")

$ run-iket --output-dir out/nb10_fmha --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fmha_iket_roles.py

[run-iket] ============= Dry run application first to collect some info... =============


/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fmha_iket_roles.py:44: DeprecationWarning: tcgen05.OperandMajorMode is deprecated, use cute.nvgpu.OperandMajorMode instead
  from cutlass.cute.nvgpu.tcgen05 import OperandMajorMode
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fmha_iket_roles.py:53: DeprecationWarning: Arch from arch.py is deprecated. Use Arch from enums.py instead.
  from cutlass.base_dsl.arch import Arch
Running Blackwell SM100 FMHA test with:
  q_shape: (1, 256, 8, 128)
  k_shape: (1, 256, 8, 128)
  qk_dtype: Float16
  pv_dtype: Float16
  out_dtype: Float16
  qk_acc_dtype: Float32
  pv_acc_dtype: Float32
  mma_tiler_mn: (128, 128)
  enable_ex2_emulation: False
  enable_skip_correction: False
  is_persistent: False
  is_causal: False
  bottom_right_align: False
  lse_calculation: False
  enable_sink: False
  window_size: (-1, -1)
  scale_q: 1.0
  scale_k: 1.0
  scale_v: 1.0
  inv_scale_o: 1.0
  scale_softmax: 0.0
  skip_softmax_threshold: N

[run-iket] Auto-computed context buffer size: 0.00 GB (14,131 bytes)
[run-iket] ============= Run application for real profiling... =============


/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fmha_iket_roles.py:44: DeprecationWarning: tcgen05.OperandMajorMode is deprecated, use cute.nvgpu.OperandMajorMode instead
  from cutlass.cute.nvgpu.tcgen05 import OperandMajorMode
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fmha_iket_roles.py:53: DeprecationWarning: Arch from arch.py is deprecated. Use Arch from enums.py instead.
  from cutlass.base_dsl.arch import Arch
Running Blackwell SM100 FMHA test with:
  q_shape: (1, 256, 8, 128)
  k_shape: (1, 256, 8, 128)
  qk_dtype: Float16
  pv_dtype: Float16
  out_dtype: Float16
  qk_acc_dtype: Float32
  pv_acc_dtype: Float32
  mma_tiler_mn: (128, 128)
  enable_ex2_emulation: False
  enable_skip_correction: False
  is_persistent: False
  is_causal: False
  bottom_right_align: False
  lse_calculation: False
  enable_sink: False
  window_size: (-1, -1)
  scale_q: 1.0
  scale_k: 1.0
  scale_v: 1.0
  inv_scale_o: 1.0
  scale_softmax: 0.0
  skip_softmax_threshold: N

[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb10_fmha/iket_pid_0x3927ce.pftrace
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb10_fmha/iket_pid_0x3927ce.trace.json
[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb10_fmha/iket_pid_0x3927ce.pftrace.gz
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb10_fmha/iket_pid_0x3927ce.html
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb10_fmha && python3 -m http.server 8080 , then open http://localhost:8080/iket_pid_0x3927ce.html
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb10_fmha/tracker
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb10_fmha/gen-config
[run-iket] Removing intermediate output directory: /path/t

0

## 4. The trace

In [5]:
nbhelp.figure("nb10/fmha0_producer_consumer_waits.png", "Producer-consumer handoffs across the attention pipeline stages.")

In [6]:
try:
    trace = ts.load(ts.find_trace("out/nb10_fmha"))
    ts.print_summary(ts.summarize(trace), top=8)
except FileNotFoundError:
    print("No local trace here — the reference numbers below were measured on a NVIDIA B300.")

kernel_cutlass_kernel___main__BlackwellFusedMultiHeadAttentionForward_object_at__TiledMMA_ThrLayoutVMNK11110000_PermutationMNK____MMAAtom_ThrID10_ShapeMNK12812816_TVLayoutA1128161281128_T_0   grid 1x8x1  block 512x1x1  warps traced 128
warp-lifetime span: 6.7 us

range                      count      median         p90       total
--------------------------------------------------------------------
correction_block              32      4.240u      4.416u     137.22u
softmax1_block                32      3.040u      3.104u      97.09u
softmax0_block                32      2.784u      2.848u      88.61u
epilogue_block                 8      5.056u      5.216u      40.70u
mma_block                      8      3.632u      3.680u      28.99u
load_block                     8      1.632u      1.632u      12.80u


## 5. The finding

Measured on a NVIDIA B300, public `fmha.py` with default shape — 128 warps, 6.66 µs span:

| Role range | Median | Count | Total |
|---|---|---|---|
| `correction_block` | **4.256 µs** | 32 | 137.28 µs |
| `softmax1_block` | 3.072 µs | 32 | 97.25 µs |
| `softmax0_block` | 2.784 µs | 32 | 88.77 µs |
| `epilogue_block` | 5.040 µs | 8 | 40.61 µs |
| `mma_block` | 3.648 µs | 8 | **28.93 µs** |
| `load_block` | 1.648 µs | 8 | 12.86 µs |


**The softmax family dominates; the matmuls do not.**

Compare totals across warps, which is the fair comparison because the roles have different warp counts:

- softmax0 + softmax1 + correction = **323.3 µs** of role time
- `mma_block` = **28.93 µs**


The softmax path costs **11× the matmul path**. The kernel is named for its attention matmuls, and they are close to free.

**Why per-warp tracing is essential here.** `mma_block` runs on one warp; softmax runs on eight, correction on four. A profiler reporting a single averaged timeline would blend a 1-warp role with a 12-warp role and tell you nothing. IKET traces each warp separately, so the roles stay distinguishable and the totals mean something.

**Corroboration from hardware metrics** (internal profiler, quoted as context): the tensor pipe runs at only ~11% of speed-of-light while the kernel is ~85% SM-throughput limited with DRAM near zero. The instruction mix is FMA and special-function heavy — `exp` and scaling. That is the softmax the trace pointed at.

IKET showed *where*; the metrics showed *why*. Reproduce the latter with `ncu --section SpeedOfLight`.

**Next:** [11 · Profile your own kernel](11_profile_your_own_kernel.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*